In [1]:
import os
import pandas as pd
import SPARQLWrapper

In [2]:
df = pd.read_csv("/Users/siddarvind/Downloads/airports.csv")
df.head()

,code,icao,name,latitude,longitude,elevation,url,time_zone,city_code,country,city,state,county,type
0,AAA,NTGA,Anaa,-17.350665,-145.511120,36,NaN,Pacific/Tahiti,AAA,PF,NaN,NaN,NaN,AP
1,AAB,YARY,Arrabury Airport,-26.696783,141.049092,328,NaN,Australia/Brisbane,AAB,AU,Tanbar,Queensland,Barcoo Shire,AP
2,AAC,HEAR,El Arish International Airport,31.074284,33.829172,85,NaN,Africa/Cairo,AAC,EG,Arish,Muhafazat Shamal Sina',NaN,AP
3,AAD,HCAD,Adado Airport,6.096286,46.637708,980,NaN,Africa/Khartoum,AAD,SO,Adado,NaN,NaN,AP
4,AAE,DABB,Les Salines Airport,36.821392,7.811857,36,NaN,Africa/Algiers,AAE,DZ,El Hadjar,Annaba,NaN,AP


In [13]:
new_df = df.dropna()
new_df['score'] = 0
new_df

/var/folders/cv/vszp_60n2yx3c1fttzx00nb40000gn/T/ipykernel_66167/717110345.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['score'] = 0


,code,icao,name,latitude,longitude,elevation,url,time_zone,city_code,country,city,state,county,type,score
11,AAL,EKYT,Aalborg Airport,57.086435,9.868076,19,http://www.aal.dk,Europe/Copenhagen,AAL,DK,Vadum,North Denmark,Alborg Kommune,AP,0
17,AAR,EKAH,Aarhus Airport,56.304543,10.612850,45,http://www.aar.dk/default.asp?id=87,Europe/Copenhagen,AAR,DK,Ebeltoft,Central Jutland,Syddjurs Kommune,AP,0
42,ABQ,KABQ,Albuquerque International Sunport,35.037669,-106.610968,5308,http://www.abqsunport.com/,America/Denver,ABQ,US,Albuquerque,New Mexico,Bernalillo County,AP,0
51,ABZ,EGPD,Aberdeen International Airport,57.201955,-2.201991,167,http://www.aberdeenairport.com/,Europe/London,ABZ,GB,Dyce,Scotland,Aberdeen City,AP,0
56,ACE,GCRR,Lanzarote Airport,28.945680,-13.607588,42,http://www.aena.es/en/lanzarote-airport/index....,Atlantic/Canary,ACE,ES,Tias,Canary Islands,Provincia de Las Palmas,AP,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9630,YYB,CYYB,Jack Garland Airport,46.356823,-79.427250,1215,https://yyb.ca/,America/Toronto,YYB,CA,North Bay,Ontario,Nipissing District,AP,0
9737,ZLO,MMZO,Manzanillo Airport,19.113333,-104.350555,30,http://manzanillo.aeropuertosgap.com.mx/index....,America/Mexico_City,ZLO,MX,Manzanillo,Colima,Manzanillo,AP,0
9764,ZRH,LSZH,Zurich Airport,47.463549,8.553205,1416,http://www.zurich-airport.com/,Europe/Zurich,ZRH,CH,Glattbrugg / Rohr/Platten-Balsberg,Zurich,Bezirk Buelach,AP,0
9769,ZSE,FMEP,St Pierre dela Reunion,-21.333332,55.483334,59,http://www.grandsudreunion.org/FR2004/aeroport...,Indian/Reunion,ZSE,RE,Saint-Pierre,Reunion,Reunion,AP,0


In [14]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time

# Step 1: Get airport coordinates using IATA code
def get_airport_coordinates(iata_code):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    query = f"""
    SELECT ?coord WHERE {{
      ?airport wdt:P238 "{iata_code}" ;
               wdt:P625 ?coord .
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    try:
        results = sparql.query().convert()
        if not results["results"]["bindings"]:
            return None
        coord_string = results["results"]["bindings"][0]["coord"]["value"]
        lon, lat = coord_string.replace('Point(', '').replace(')', '').split()
        return float(lat), float(lon)
    except Exception as e:
        print(f"Error fetching coordinates for {iata_code}: {e}")
        return None

# Step 2: Get nearby interesting places using coordinates
def get_nearby_places(lat, lon, radius_km=50):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    query = f"""
    SELECT ?placeLabel ?typeLabel ?coord WHERE {{
      SERVICE wikibase:around {{
        ?place wdt:P625 ?coord .
        bd:serviceParam wikibase:center "Point({lon} {lat})"^^geo:wktLiteral .
        bd:serviceParam wikibase:radius "{radius_km}" .
      }}
      ?place wdt:P31 ?type .
      FILTER(?type IN (
        wd:Q570116,    # tourist attraction
        wd:Q33506,     # museum
        wd:Q1549591,   # financial district
        wd:Q515,       # city
        wd:Q3918,      # university
        wd:Q1473346,   # UNESCO World Heritage Site
        wd:Q811979,    # architectural structure
        wd:Q4989906    # economic zone
      ))
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 100
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    try:
        results = sparql.query().convert()
        data = []
        for item in results["results"]["bindings"]:
            data.append({
                "Place": item["placeLabel"]["value"],
                "Type": item["typeLabel"]["value"],
                "Coordinates": item["coord"]["value"]
            })
        return data
    except Exception as e:
        print(f"Error fetching places near ({lat}, {lon}): {e}")
        return []

# Step 3: Process a list of IATA codes
def analyze_airports(iata_codes, delay=1.5):
    all_data = []
    for iata in iata_codes:
        print(f"Processing {iata}...")
        coords = get_airport_coordinates(iata)
        if coords:
            places = get_nearby_places(*coords)
            for place in places:
                all_data.append({
                    "IATA": iata,
                    "Place": place["Place"],
                    "Type": place["Type"],
                    "Coordinates": place["Coordinates"]
                })
        else:
            print(f"Coordinates not found for {iata}")
        time.sleep(delay)  # respectful delay to avoid rate-limiting
    return pd.DataFrame(all_data)

# Example usage:
for iata in new_df.iloc[:, 0]:
    airport_insights_df = analyze_airports([iata]) 
    new_df['score'] = airport_insights_df.shape[0]
new_df.head(40)

Processing AAL...


/var/folders/cv/vszp_60n2yx3c1fttzx00nb40000gn/T/ipykernel_66167/63659239.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['score'] = len(airport_insights_df)


Processing AAR...
Processing ABQ...
Processing ABZ...
Processing ACE...
Processing ACY...
Processing ADW...
Processing AES...
Processing AFN...
Processing AFO...


/var/folders/cv/vszp_60n2yx3c1fttzx00nb40000gn/T/ipykernel_66167/63659239.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['score'] = len(airport_insights_df)


Processing AFW...
Processing AGB...
Processing AGU...
Processing AHO...
Processing AJL...
Processing AJU...
Processing AKL...
Processing AKO...
Processing ALB...
Processing ALF...
Processing ALL...
Processing ALS...
Processing AMN...
Processing AMS...
Processing AMZ...
Processing ANR...
Processing ANX...
Processing AOI...
Processing AOT...
Processing APC...
Processing AQA...
Processing ARA...
Processing ARG...
Processing ARN...
Processing ARU...
Processing ASE...
Processing ATH...
Processing ATQ...
Processing ATW...
Processing AUS...
Processing AUW...
Processing AVB...
Processing AVP...


KeyboardInterrupt: 

In [12]:
new_df

,IATA,Place,Type,Coordinates
0,AAL,Johannes V. Jensen Museum,museum,Point(9.33978 56.76804)
1,AAL,Regan Vest,museum,Point(9.8018 56.8245)
2,AAL,The Museum Society of Hadsund,museum,Point(10.11487 56.7236)
3,AAL,Q19721224,museum,Point(10.11452 56.72347)
4,AAL,Hadsund Command Centre,museum,Point(10.1068 56.7289)
...,...,...,...,...
593,ALB,National Bottle Museum,museum,Point(-73.848666666 43.003111111)
594,ALB,National Museum of Racing and Hall of Fame,museum,Point(-73.7734 43.0763)
595,ALB,Saratoga Springs History Museum,museum,Point(-73.7841 43.0783)
596,ALB,Frances Young Tang Teaching Museum and Art Gal...,museum,Point(-73.7859 43.0954)
